# Farmer's Market Produce & Blood Pressure Study

**Objective:** farmer's-market produce boxes are being used more and more as a
"food as medicine" intervention for high blood pressure. What's less clear is
whether just handing someone produce does anything on its own, or whether it
needs guidance attached to it, and if so, whether that guidance has to come
from a person.

This notebook compares three groups getting the same produce box:

- **Dietitian** - one-on-one human counseling
- **AI Assistant** - an AI-delivered nutrition assistant
- **Control** - produce only, no guidance

**Data:** 300 participants, 100 per group. Baseline demographics and starting
blood pressure are sampled from a real 70,000-record cardiovascular dataset;
the study design on top of that (group assignment, produce baskets, adherence,
follow-up BP) is simulated, since no real trial of this exact design exists yet.

**What this notebook covers:** load -> explore -> clean -> visualize -> test the
group differences statistically -> build a predictive model -> summarize what
actually drives blood pressure change. Expected output is a clear read on
whether guidance matters, and what's really behind the effect if it does.

## Import Libraries

Kept to what's actually used below - core data handling, stats for the
hypothesis tests, sklearn/xgboost for the models, plotting last.

In [ ]:
# data handling
import numpy as np
import pandas as pd

# stats
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# modeling
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import cross_validate, train_test_split, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

# plotting
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk", font_scale=0.75)
RANDOM_STATE = 42

## Load the Data

Reads straight from the dataset's home in the GitHub repo, so this cell
just runs, no upload step, no re-uploading next time.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/edefang/farmers-market-bp-study/main/data/bp_farmers_market_study_dataset.csv"
df = pd.read_csv(DATA_URL)
df.shape

In [ ]:
# quick gut check that this is actually the right file before going further
df.head()

## Data Exploration

Before touching the research question, worth confirming the data is what it
claims to be: right shape, sane types, nothing missing or duplicated.

In [ ]:
df.dtypes.value_counts()

42 numeric columns (mostly the produce baskets), 10 text categories, a couple of ints and booleans. Nothing here needs recasting.

In [ ]:
df.isna().sum().sum()

Zero missing cells across all 57 columns. One less thing to worry about.

In [ ]:
df.duplicated().sum(), df['participant_id'].duplicated().sum()

No duplicate rows and no repeated participant IDs, so each row really is one person.

In [ ]:
df[['baseline_systolic_bp', 'baseline_diastolic_bp', 'systolic_change', 'diastolic_change', 'adherence_pct']].describe()

Systolic and diastolic change both average negative, so BP dropped for
most people over the study. Adherence ranges the full 0-100%, which matters
since that's the variable this whole analysis ends up circling back to.

In [ ]:
df['group'].value_counts()

Even split, 100 per arm, as designed.

## Data Cleaning and Preparation

The summary stats above hid something the `describe()` call on
`baseline_diastolic_bp` should have flagged: a max value of 1000. Real
diastolic blood pressure doesn't get anywhere near that. Digging in below.

In [ ]:
df['baseline_diastolic_bp'].describe()

In [ ]:
suspect = df[(df['baseline_diastolic_bp'] > 250) | (df['baseline_systolic_bp'] < 60)]
suspect[['participant_id', 'baseline_systolic_bp', 'baseline_diastolic_bp',
         'followup_systolic_bp', 'followup_diastolic_bp']]

Four rows, two separate issues. Three participants show a baseline
diastolic of 1000 with a follow-up around 997-998, almost certainly a stray
zero added somewhere upstream (100 -> 1000) rather than three people
independently getting the same impossible reading. The fourth has a baseline
systolic of 17, which isn't a survivable reading either. These read like data
entry artifacts in the source file, not real extreme physiology, so dropping
them is more honest than trying to guess the "correct" value.

In [ ]:
before = len(df)
df = df[~((df['baseline_diastolic_bp'] > 250) | (df['baseline_systolic_bp'] < 60))].reset_index(drop=True)
print(f"dropped {before - len(df)} rows, {len(df)} remain")

Cholesterol and glucose are already stored as `Normal / High / Extremely
High` text categories rather than numeric codes, so nothing to fix there -
they'll go through one-hot encoding later as-is. No other dtype or outlier
issues turned up in the rest of the columns.

## Exploratory Data Analysis

Four charts here, each aimed at a specific question rather than just
plotting everything available.

In [ ]:
GROUP_ORDER = ['Control_NoGuidance', 'Dietitian', 'AI_Assistant']
GROUP_COLORS = {'Control_NoGuidance': '#8c8c8c', 'Dietitian': '#2E86AB', 'AI_Assistant': '#E27D60'}
TARGETS = ['systolic_change', 'diastolic_change']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, target, label in zip(axes, TARGETS, ['Systolic BP Change (mmHg)', 'Diastolic BP Change (mmHg)']):
    sns.boxplot(data=df, x='group', y=target, order=GROUP_ORDER, hue='group',
                palette=[GROUP_COLORS[g] for g in GROUP_ORDER], legend=False, ax=ax)
    ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.6)
    ax.set_xlabel('')
    ax.set_ylabel(label)
fig.suptitle('Blood Pressure Change by Study Arm', y=1.03)
plt.show()

Control sits right on the zero line, Dietitian drops the furthest, AI
Assistant lands in between. This is the headline result of the whole project
in one picture: guidance matters, and human guidance edges out AI guidance,
but AI isn't doing nothing either. Everything after this chart is really
about explaining why this pattern shows up.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for g in GROUP_ORDER:
    sub = df[df['group'] == g]
    ax.scatter(sub['adherence_pct'], sub['systolic_change'], label=g.replace('_', ' '),
               color=GROUP_COLORS[g], alpha=0.7, edgecolor='white', linewidth=0.4)
ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Adherence (% of provided produce eaten)')
ax.set_ylabel('Systolic BP Change (mmHg)')
ax.set_title('The Adherence Confound: BP Change vs. Adherence, by Group')
ax.legend(title='Group')
plt.show()

Dietitian points cluster on the high-adherence side, Control clusters
low. That's the confound this whole study design has to deal with: guided
groups don't just get guidance, they also end up eating more of what they're
given. Before crediting "guidance" for the group differences above, adherence
needs to be controlled for statistically, which is exactly what the ANCOVA
below is for.

In [ ]:
age_order = sorted(df['age_range'].unique(), key=lambda s: int(s.split('-')[0]))
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='age_range', y='baseline_systolic_bp', order=age_order, color='#2E86AB', ax=ax)
ax.set_xlabel('Age Range')
ax.set_ylabel('Baseline Systolic BP (mmHg)')
ax.set_title('Baseline Systolic BP by Age Range')
plt.show()

Baseline BP climbs gradually with age, which is exactly what you'd
expect in a real population sample and a good sign the cleaning step above
didn't leave anything broken. No age bucket looks out of place, so age isn't
a hidden confound sitting underneath the group comparison.

In [ ]:
cols = ['adherence_pct', 'total_produce_kg_provided', 'guidance_sessions_or_interactions',
        'baseline_systolic_bp', 'baseline_diastolic_bp', 'bmi', 'systolic_change', 'diastolic_change']
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Correlation Heatmap: Key Variables')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.show()

Adherence is the only variable with a strong pull on either outcome;
everything else in this matrix is pale by comparison. Total produce provided
barely correlates with the change columns at all, which is a useful finding
on its own - handing out more food isn't what matters, eating it is.

## Statistical Analysis and Modeling

Two questions to answer here: is the group difference real once adherence is
accounted for, and can BP change be predicted well enough to say what's
actually driving it.

### Hypothesis testing

One-way ANOVA first (is there a group difference at all), Kruskal-Wallis as a
non-parametric cross-check, then ANCOVA adding adherence and baseline BP as
covariates. ANCOVA is the one that actually answers the confound question from
the scatterplot above: if group is still significant after adherence is in
the model, guidance is doing something beyond just driving adherence up.

In [ ]:
for target in TARGETS:
    groups = [g[target].values for _, g in df.groupby('group')]
    f_stat, p_anova = stats.f_oneway(*groups)
    h_stat, p_kw = stats.kruskal(*groups)
    print(f"{target}: ANOVA F={f_stat:.2f} p={p_anova:.2e}  |  Kruskal-Wallis H={h_stat:.2f} p={p_kw:.2e}")

In [ ]:
for target in TARGETS:
    baseline_col = 'baseline_systolic_bp' if 'systolic' in target else 'baseline_diastolic_bp'
    model = ols(f'{target} ~ C(group) + adherence_pct + {baseline_col}', data=df).fit()
    table = sm.stats.anova_lm(model, typ=2)
    print(f"{target}: group p={table.loc['C(group)', 'PR(>F)']:.2e}  "
          f"adherence p={table.loc['adherence_pct', 'PR(>F)']:.2e}  R2={model.rsquared:.3f}")

Group stays significant well past adherence being in the model (p <
0.0001 both outcomes). So the boxplot pattern isn't just adherence wearing a
disguise - guidance type is doing something on top of it.

### Predictive modeling

Four models, same preprocessing pipeline, 5-fold CV so the comparison isn't
riding on one lucky train/test split with only ~300 rows to work with.

In [ ]:
LEAK_OR_ID_COLS = ['participant_id', 'followup_systolic_bp', 'followup_diastolic_bp',
                    'systolic_change', 'diastolic_change', 'assumption_diet_restricted_to_provided_produce']
CATEGORICAL_COLS = ['group', 'age_range', 'sex', 'race', 'cholesterol', 'glucose',
                     'smoke', 'alcohol_intake', 'physical_activity', 'ate_provided_food']

MODELS = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
}

def split_features(target):
    # followup BP would leak the answer (it's just baseline + change), so it's out
    X = df.drop(columns=[c for c in LEAK_OR_ID_COLS if c != target] + [target])
    y = df[target]
    cat_cols = [c for c in CATEGORICAL_COLS if c in X.columns]
    num_cols = [c for c in X.columns if c not in cat_cols]
    return X, y, num_cols, cat_cols

def make_prep(num_cols, cat_cols):
    return ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ])

def cv_compare(target):
    X, y, num_cols, cat_cols = split_features(target)
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for name, model in MODELS.items():
        pipe = Pipeline([('prep', make_prep(num_cols, cat_cols)), ('model', model)])
        scores = cross_validate(pipe, X, y, cv=cv, scoring={
            'r2': 'r2', 'neg_rmse': 'neg_root_mean_squared_error', 'neg_mae': 'neg_mean_absolute_error',
        })
        rows.append({'model': name, 'r2_mean': scores['test_r2'].mean(), 'r2_std': scores['test_r2'].std(),
                     'rmse_mean': -scores['test_neg_rmse'].mean(), 'mae_mean': -scores['test_neg_mae'].mean()})
    return pd.DataFrame(rows).sort_values('r2_mean', ascending=False).reset_index(drop=True)

In [ ]:
systolic_cv = cv_compare('systolic_change')
systolic_cv

In [ ]:
diastolic_cv = cv_compare('diastolic_change')
diastolic_cv

Random Forest comes out on top for both targets, though the gap over
the linear baselines is bigger on systolic than diastolic. Diastolic change
is just a harder target to predict here across the board.

In [ ]:
def evaluate_holdout(target, winner_name):
    X, y, num_cols, cat_cols = split_features(target)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
    pipe = Pipeline([('prep', make_prep(num_cols, cat_cols)), ('model', MODELS[winner_name])])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    return pipe, {'r2': r2_score(y_test, y_pred), 'rmse': mean_squared_error(y_test, y_pred) ** 0.5,
                  'mae': mean_absolute_error(y_test, y_pred)}

systolic_winner = systolic_cv.iloc[0]['model']
diastolic_winner = diastolic_cv.iloc[0]['model']

systolic_pipe, systolic_metrics = evaluate_holdout('systolic_change', systolic_winner)
diastolic_pipe, diastolic_metrics = evaluate_holdout('diastolic_change', diastolic_winner)

print('systolic:', systolic_winner, systolic_metrics)
print('diastolic:', diastolic_winner, diastolic_metrics)

In [ ]:
def plot_importance(pipe, target, winner_name):
    prep = pipe.named_steps['prep']
    num_cols = prep.transformers_[0][2]
    cat_names = prep.transformers_[1][1].get_feature_names_out(prep.transformers_[1][2])
    feature_names = list(num_cols) + list(cat_names)

    model = pipe.named_steps['model']
    importances = model.feature_importances_ if hasattr(model, 'feature_importances_') else np.abs(model.coef_)
    top = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(9, 7))
    top.sort_values().plot.barh(ax=ax, color='#2E86AB')
    ax.set_title(f'Top 15 Features - {winner_name} ({target})')
    plt.show()
    return top

systolic_top = plot_importance(systolic_pipe, 'systolic_change', systolic_winner)

Adherence isn't just the top feature, it's in a different league from
everything else on this chart. That lines up with the correlation heatmap
earlier and with the ANCOVA result - three separate methods, same answer.

In [ ]:
diastolic_top = plot_importance(diastolic_pipe, 'diastolic_change', diastolic_winner)

Same story on diastolic. Whatever target you pick, adherence is doing most of the work.

### Is the adherence relationship actually linear?

The scatterplot earlier looked roughly straight, but "looked straight" isn't
a great way to decide a model's shape. Fitting linear against a couple of
polynomial degrees and checking cross-validated R2, not just the R2 on the
same data the curve was fit to, settles it properly.

In [ ]:
x = df['adherence_pct'].values.reshape(-1, 1)
y = df['systolic_change'].values

for degree in [1, 2, 3]:
    poly_model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    poly_model.fit(x, y)
    cv_scores = cross_validate(poly_model, x, y, cv=KFold(5, shuffle=True, random_state=RANDOM_STATE), scoring='r2')['test_score']
    print(f"degree {degree}: train R2={poly_model.score(x, y):.3f}  CV R2={cv_scores.mean():.3f}")

Going from a straight line to a curve barely moves the training R2 and
doesn't help the cross-validated score at all - degree 2 and 3 are just
fitting noise. Adherence vs. systolic change is a straight-line relationship;
no need to overcomplicate it.

In [ ]:
slope, intercept, r_value, p_value, std_err = stats.linregress(df['adherence_pct'], df['systolic_change'])

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(df['adherence_pct'], df['systolic_change'], color='#555555', alpha=0.6, edgecolor='white', linewidth=0.4)
x_line = np.linspace(df['adherence_pct'].min(), df['adherence_pct'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, color='black', linewidth=2)
ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.4)
ax.set_xlabel('Adherence (%)')
ax.set_ylabel('Systolic BP Change (mmHg)')
ax.set_title(f'y = {slope:.3f}x + {intercept:.2f}   (R2 = {r_value**2:.3f}, p = {p_value:.1e})')
plt.show()

Roughly a tenth of a point of systolic drop for every point of adherence gained. Small per-unit effect, but adherence ranges the full 0-100, so it adds up to the biggest lever in the dataset.

## Results and Interpretation

Going back to the original question: does guidance matter, and does it
matter who or what delivers it?

**Yes to both, with caveats.** Dietitian and AI Assistant both beat Control on
BP reduction, and that gap survives the ANCOVA adjustment for adherence, so
it isn't just a story about guided people eating more of their box. Something
about the guidance itself, maybe which items get eaten, maybe behavior this
dataset doesn't capture, is adding value beyond raw adherence.

At the same time, adherence is clearly the single biggest lever in the whole
analysis: strongest correlation, strongest ANCOVA covariate, strongest model
feature, by a wide margin every time. AI Assistant lands between Dietitian
and Control on both adherence and outcome, so it's picking up part of the
Dietitian effect without fully closing the gap.

A couple of things that stood out that weren't part of the original
hypothesis: individual produce items barely correlate with outcome at all -
no single vegetable matters, it's about the percentage eaten, not what's in
the box. And diastolic change was consistently harder to predict than
systolic across every model tried, worth flagging for anyone building on
this later.

## Conclusion

Guided nutrition support, whether from a dietitian or an AI assistant, is
associated with meaningfully larger blood pressure drops than just handing
someone produce and hoping for the best, and that holds up even after
controlling for how much of the produce actually got eaten. Adherence is the
strongest single driver of outcome found anywhere in this analysis, and the
relationship between adherence and BP change is a clean straight line, not
something that needed a fancier model to describe.

**Limitations:** the outcome data (group assignment, adherence, follow-up BP)
is simulated on top of a real baseline population, so this is a methods
demonstration, not a clinical result. Four rows with clearly broken baseline
BP readings were dropped during cleaning, a defensible call given how
physiologically implausible those values were, but it's a judgment call worth
being transparent about. With under 300 rows and dozens of features, the
individual produce-item findings especially shouldn't be trusted very far.

**What I'd want to add next:** the biggest gap is that everything here is
predictive, not causal - a proper causal forest (subgroup treatment-effect
estimation) would say whether AI guidance helps some people more than others,
rather than just ranking which features predict the outcome best. And
obviously, the real next step is running this design on an actual trial
rather than a synthetic outcome layer.